In [1]:
import sys
import os

# Tambahkan path src ke sys.path
sys.path.append(os.path.abspath("..")) 

In [5]:

import torch
import numpy as np
import torch.nn.functional as F
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from types import SimpleNamespace
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from src.model.mnist_cban_generator import Net, hiddenNet, train, convertToOneHotEncoding, insertSingleBD, transformImg

def test(
    args, model, device, test_loader, bdModel,
    save_dir="saved_backdoor_images",
    visualize=False,      # new flag
    max_images=25        # jumlah sampel untuk divisualisasi
):
    model.eval()
    bdModel.eval()
    criterion = torch.nn.NLLLoss(reduction='sum')

    nz, numClasses, BDSize = 100, 10, 5
    collected = []  # list of (tensor1x28x28, true, pred)
    os.makedirs(save_dir, exist_ok=True)

    test_loss, test_lossBD = 0, 0
    correct, correctBD = 0, 0
    correctBDtotal, BDlosstotal = 0, 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            batch_size = data.size(0)

            # ——— Clean evaluation ———
            out = model(transformImg(data))
            test_loss += criterion(out, target).item()
            pred = out.argmax(dim=1)
            correct += pred.eq(target).sum().item()

            # ——— Backdoor evaluation ———
            for i in range(numClasses):
                noise = torch.rand(batch_size, nz, device=device)
                targetBD = torch.ones(batch_size).long().to(device)*i
                oh = convertToOneHotEncoding(targetBD)
                bd_patches = bdModel(oh, noise).view(-1,1,BDSize,BDSize)
                dataBD = insertSingleBD(data, bd_patches, i)
                
                outBD = model(dataBD)
                test_lossBD = F.nll_loss(outBD, targetBD, reduction='sum').item()  
                predBD = outBD.argmax(dim=1)
                correctBD = predBD.eq(targetBD).sum().item()
                correctBDtotal += correctBD            
                    
                BDlosstotal += test_lossBD
                print('Class ' + str(i))
                print('\nBackDoor Test set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
                test_lossBD, correctBD, len(test_loader.dataset),
                100. * correctBD / len(test_loader.dataset)))

    test_loss /= len(test_loader.dataset)
    
    mean = correctBDtotal / 10 / 100
    avgBDlosstotal = BDlosstotal / 10
    
    print(f'Backdoor set (avg over all classes): '
          f'Avg loss: {avgBDlosstotal:.4f}, '
          f'Accuracy: {mean} %')

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))


# cell 2: setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Data loaders MNIST
transform = transforms.Compose([transforms.ToTensor()])
train_loader = DataLoader(
    datasets.MNIST('../data', train=True, download=True, transform=transform),
    batch_size=64, shuffle=True)
test_loader  = DataLoader(
    datasets.MNIST('../data', train=False, transform=transform),
    batch_size=10000, shuffle=False)

args = SimpleNamespace(
    batch_size=64,
    test_batch_size=10000,
    log_interval=100,
    save_model=False,
    output_dir='bdImages',

)

model   = Net().to(device)
bdModel = hiddenNet().to(device)

optimizer   = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizerBD = torch.optim.Adam(bdModel.parameters(), lr=1e-3)

for epoch in range(1, 16):
        train(args, model, device, train_loader, optimizer, epoch, bdModel, optimizerBD)
        test(args, model, device, test_loader, bdModel)
        # torch.save(model.state_dict(), "models/mnist_cnn.pth")

Train Epoch: 1 [0/60000 (0%)]	LossBD: 23.057798
Train Epoch: 1 [6400/60000 (11%)]	LossBD: 18.809757
Train Epoch: 1 [12800/60000 (21%)]	LossBD: 17.209274
Train Epoch: 1 [19200/60000 (32%)]	LossBD: 17.517010
Train Epoch: 1 [25600/60000 (43%)]	LossBD: 16.948074
Train Epoch: 1 [32000/60000 (53%)]	LossBD: 16.919477
Train Epoch: 1 [38400/60000 (64%)]	LossBD: 16.647169
Train Epoch: 1 [44800/60000 (75%)]	LossBD: 16.670034
Train Epoch: 1 [51200/60000 (85%)]	LossBD: 16.741419
Train Epoch: 1 [57600/60000 (96%)]	LossBD: 16.089825
Class 0

BackDoor Test set: Average loss: 15950.7031, Accuracy: 2216/10000 (22%)

Class 1

BackDoor Test set: Average loss: 12384.5488, Accuracy: 5479/10000 (55%)

Class 2

BackDoor Test set: Average loss: 16077.9150, Accuracy: 1987/10000 (20%)

Class 3

BackDoor Test set: Average loss: 15668.4355, Accuracy: 4989/10000 (50%)

Class 4

BackDoor Test set: Average loss: 15904.5752, Accuracy: 2414/10000 (24%)

Class 5

BackDoor Test set: Average loss: 16410.3008, Accuracy: 13

KeyboardInterrupt: 